# 🎓 CKP03 — Tutor ENEM · Agente Inteligente

**FIAP · Prompt Engineering and Artificial Intelligence · 2º semestre · Checkpoint 03 (Módulo 3)**
Prof. Jorge Luiz Gomes · Turma 1CCPK

| Integrante | RM |
|---|---|
| Daniel Vilela Mana | 571632 |
| Davi Queiroz Zuolo | 571669 |
| Gustavo Zagato Bottechia | 569420 |
| Kayo Henderson | 570706 |

**Domínio:** preparação para o **ENEM 2026** (provas em 8/11 e 15/11/2026). O estudante escolhe a matéria —
✍️ Redação · 📐 Matemática · 🌎 História & Geografia · 🧬 Biologia · ⚡ Física — e conversa com um **agente
especializado só naquela matéria**. Todos os agentes seguem as mesmas regras; muda a persona e o kit de tools.

**Arco do semestre:** CKP01 (chatbot: system prompt + memória + Pydantic) → CKP02 (pipeline RAG medido com RAGAS) →
**CKP03 (agente ReAct que decide sozinho quando usar cada tool)**. Do CKP02 reaproveitamos o **código do pipeline**
(load → split → embed → store → retrieve + filtro de metadados + reranking) e a **configuração vencedora do RAGAS**
(Recursive-1024), agora sobre uma base 100% oficial do ENEM (INEP).

### Mapa do notebook → rubrica do CKP03
| Seção | O que entrega | Critério |
|---|---|---|
| 1–2 | Instalação, `OLLAMA_API_KEY` como Secret, `gemma4:cloud` via Ollama Cloud | Requisito técnico |
| 3–5 | Base oficial do ENEM (INEP) → chunks → ChromaDB com metadado `materia` | RAG do CKP02 (2,5) |
| 6 | **RAG como tool** com `create_retriever_tool()`, filtrado por matéria, citando fonte e página | RAG como tool (2,5) |
| 7 | Tools de domínio (`@tool`) com descriptions comentadas | Tools + descriptions (1,5) |
| 8 | Agentes ReAct (`AgentExecutor`) — 1 por matéria, memória por sessão | Agente funcional (4,0) |
| 9 | Guardrail de input + compressão de contexto | Diferenciais (+1,0) |
| 10 | 3 interações documentadas (RAG · outra tool · sem tool) | Agente funcional (4,0) |
| 11 | Interface Gradio com histórico, botão limpar e `share=True` | Gradio + URL (2,0) |

### ⚙️ Pré-requisito para executar
1. Crie uma chave gratuita em <https://ollama.com> (*Settings → API keys*).
2. No Colab, ícone 🔑 **Secrets** → *Add new secret* → nome `OLLAMA_API_KEY`, cole a chave e **ative o acesso a este notebook**.
3. *Runtime → Run all*. O notebook clona a base do GitHub, sobe o Ollama local (embeddings), indexa e publica a interface.

## 1. Instalação das dependências

> **Por que `langchain-classic`?** A Aula 11 importa `AgentExecutor`, `create_react_agent`, `hub` e
> `create_retriever_tool` de `langchain.*`. No LangChain 1.x (versão que o Colab instala hoje) essas peças saíram do
> pacote `langchain` e passaram para o **`langchain-classic`** — **mesma API, mesmo comportamento**, só muda o import.
> Com o import antigo a célula quebra com `ImportError`.

> Se o Colab pedir *Restart session* depois da instalação, reinicie e rode *Run all* de novo.

In [ ]:
!pip install -q langchain langchain-classic langchain-core langchain-ollama langchain-community langchain-chroma langchain-text-splitters chromadb pymupdf sentence-transformers gradio pandas wikipedia tiktoken sympy

## 2. Configuração do projeto

Toda a configuração fica num lugar só. O dicionário **`MATERIAS`** é o "registro" do sistema: cada agente, cada filtro
do RAG e cada opção da interface sai dele. Para adicionar uma 6ª matéria no futuro basta uma nova entrada aqui
(e os documentos dela no `fontes.csv`) — nenhuma outra célula precisa mudar.

In [ ]:
import os, re, json, time, shutil, zipfile, hashlib, subprocess, unicodedata, warnings
from datetime import date
from pathlib import Path

import pandas as pd
import fitz  # PyMuPDF
import chromadb
from IPython.display import display, Markdown

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 120)

# ===================== CONFIGURAÇÃO DO PROJETO =====================
DOMINIO = "Tutor ENEM"

# Repositório público com docs/, dados/ e fontes.csv — o professor roda o "Run All" sem enviar arquivo nenhum.
# Deixe "" para enviar os arquivos manualmente pelo upload do Colab.
REPO_GIT = "https://github.com/DaviZuolo07/Agente_Inteligente.git"

try:
    import google.colab  # noqa: F401  (só existe dentro do Colab)
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
PASTA_PROJETO = Path("/content/tutor_enem") if EM_COLAB else Path.cwd() / "tutor_enem"
PASTA_DOCS    = PASTA_PROJETO / "docs"                          # PDFs oficiais (INEP)
ARQ_FONTES    = PASTA_PROJETO / "fontes.csv"                    # manifesto: título, tipo, matéria, data e link
ARQ_QUESTOES  = PASTA_PROJETO / "dados" / "banco_questoes_enem.json"   # questões reais extraídas das provas
PASTA_CHROMA  = Path("/content/chroma_tutor_enem") if EM_COLAB else Path.cwd() / "chroma_tutor_enem"

# Modelos (CKP03: chat EXCLUSIVAMENTE gemma4:cloud via Ollama Cloud)
OLLAMA_HOST      = "https://ollama.com"
MODELO_LLM       = "gemma4:cloud"
MODELO_EMBEDDING = "nomic-embed-text"
MODELO_RERANKER  = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Chunking: configuração VENCEDORA do RAGAS no CKP02 (Recursive-1024: faithfulness 0,771 vs 0,698 do 512)
CHUNK_SIZE           = 1024
CHUNK_OVERLAP        = 128          # 12,5%
SEPARADORES          = ["\n\n", "\n", ". ", " ", ""]
TAMANHO_MINIMO_CHUNK = 60           # descarta números de página e rótulos soltos

# Recuperação
K_FINAL      = 4    # trechos entregues ao agente (menor que no CKP02: cada Observation entra no scratchpad do ReAct)
K_CANDIDATOS = 12   # candidatos buscados antes do reranking

# ===================== AS 5 MATÉRIAS (registro único) =====================
# "geral" = documento que vale para todas as matérias (Matriz de Referência, Edital).
MATERIAS = {
    "redacao":    {"nome": "Redação",              "emoji": "✍️", "dia": 1,
                   "area": "Linguagens, Códigos e suas Tecnologias — Redação"},
    "matematica": {"nome": "Matemática",           "emoji": "📐", "dia": 2,
                   "area": "Matemática e suas Tecnologias"},
    "hist_geo":   {"nome": "História & Geografia", "emoji": "🌎", "dia": 1,
                   "area": "Ciências Humanas e suas Tecnologias"},
    "biologia":   {"nome": "Biologia",             "emoji": "🧬", "dia": 2,
                   "area": "Ciências da Natureza e suas Tecnologias"},
    "fisica":     {"nome": "Física",               "emoji": "⚡", "dia": 2,
                   "area": "Ciências da Natureza e suas Tecnologias"},
}
MATERIA_GERAL = "geral"

# Datas oficiais do ENEM 2026 (Edital INEP): usadas pelo planejador de estudos
DATAS_ENEM_2026 = {
    1: date(2026, 11, 8),    # Linguagens + Redação + Ciências Humanas (5h30)
    2: date(2026, 11, 15),   # Ciências da Natureza + Matemática (5h)
}

print(f"Domínio: {DOMINIO} | Colab: {EM_COLAB} | Matérias: {', '.join(m['nome'] for m in MATERIAS.values())}")

### 2.1 Segredos, LLM e embeddings

- **LLM:** `gemma4:cloud` no **Ollama Cloud**, `temperature=0` (saída mais previsível, menos `OutputParserException` no
  ReAct). A chave vem dos *Secrets* do Colab — nunca fica escrita no código.
- **Embeddings:** mesmo arranjo do CKP02. O endpoint `/api/embed` do Ollama Cloud devolve `401` mesmo com chave válida,
  então o **mesmo `nomic-embed-text`** roda num Ollama local dentro do Colab (função `iniciar_ollama_local()`, código
  reaproveitado do CKP02 sem alteração).

In [ ]:
def obter_segredo(nome: str) -> str:
    """Lê um segredo do Colab (Secrets) ou, fora do Colab, de uma variável de ambiente."""
    valor = None
    try:
        from google.colab import userdata
        valor = userdata.get(nome)
    except Exception:
        valor = os.environ.get(nome)
    if not valor:
        raise RuntimeError(f"Segredo '{nome}' não encontrado. Ícone 🔑 → Secrets → '{nome}' com acesso liberado.")
    return valor.strip()


import httpx

# LLM no Ollama Cloud (gemma4:cloud). A chave vai como Secret.
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
os.environ["OLLAMA_API_KEY"] = obter_segredo("OLLAMA_API_KEY")
llm = ChatOllama(model=MODELO_LLM, temperature=0, base_url=OLLAMA_HOST)

# Embeddings: o Ollama Cloud recusa /api/embed (401) mesmo com chave válida.
# Então o MESMO nomic-embed-text roda num servidor Ollama local dentro do Colab.
EMB_HOST = "http://127.0.0.1:11434"
OLLAMA_VERSAO = ""                      # "" = versão mais recente; ex.: "0.35.1" para fixar
PREFIXO_OLLAMA = Path("/usr/local")     # o pacote traz bin/ollama e lib/ollama/ (onde fica o llama-server)
PACOTE_OLLAMA = "ollama-linux-amd64"


def localizar_llama_server() -> Path | None:
    """Procura o llama-server, o binário que carrega o modelo (separado do `ollama` desde a versão 0.30)."""
    pasta = PREFIXO_OLLAMA / "lib" / "ollama"
    return next(iter(sorted(pasta.rglob("llama-server"))), None) if pasta.exists() else None


def ollama_instalado_por_inteiro() -> bool:
    return (PREFIXO_OLLAMA / "bin" / "ollama").exists() and localizar_llama_server() is not None


def _rodar(comando: str, oque: str) -> None:
    """Executa um comando de shell e, se falhar, mostra o erro em vez de escondê-lo."""
    r = subprocess.run(["bash", "-c", f"set -o pipefail; {comando}"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{oque} falhou (código {r.returncode}):\n{(r.stderr or r.stdout)[-1200:]}")


def _url_existe(url: str) -> bool:
    return subprocess.run(["curl", "--fail", "--silent", "--head", "--location", "--max-time", "30", url],
                          capture_output=True).returncode == 0


def fontes_do_pacote() -> list[tuple[str, str]]:
    """URLs candidatas, na ordem de tentativa: site oficial e, de reserva, o release no GitHub.
    As versões atuais usam .tar.zst; o .tgz só existe em versões antigas."""
    versao = OLLAMA_VERSAO.lstrip("v")
    sufixo = f"?version={versao}" if versao else ""
    github = (f"https://github.com/ollama/ollama/releases/download/v{versao}" if versao
              else "https://github.com/ollama/ollama/releases/latest/download")
    return [(f"{base}/{PACOTE_OLLAMA}{ext}{suf}", ext)
            for base, suf in (("https://ollama.com/download", sufixo), (github, ""))
            for ext in (".tar.zst", ".tgz")]


def instalar_ollama() -> None:
    """Baixa o pacote oficial do Ollama, extrai em /usr/local e confere se veio completo."""
    print("📦 Instalando o Ollama local (pacote de ~1,4 GB, 1–3 min)…")
    subprocess.run(["pkill", "-f", "ollama serve"])                     # derruba um servidor de instalação anterior
    time.sleep(1)
    shutil.rmtree(PREFIXO_OLLAMA / "lib" / "ollama", ignore_errors=True)
    (PREFIXO_OLLAMA / "bin" / "ollama").unlink(missing_ok=True)
    if shutil.which("zstd") is None:                                    # necessário para extrair o .tar.zst
        _rodar("apt-get -qq update && apt-get -qq install -y zstd", "Instalação do zstd")

    erros = []
    for url, ext in fontes_do_pacote():
        if not _url_existe(url):
            erros.append(f"não encontrado: {url}")
            continue
        arquivo = Path(f"/tmp/{PACOTE_OLLAMA}{ext}")
        try:
            # Baixa para um arquivo antes de extrair: um download interrompido falha aqui,
            # em vez de deixar uma instalação pela metade (binário sem llama-server).
            _rodar(f"curl --fail --location --silent --show-error --retry 3 --retry-delay 3 -o '{arquivo}' '{url}'",
                   "Download do Ollama")
            extrair = (f"zstd -d -c '{arquivo}' | tar -xf - -C '{PREFIXO_OLLAMA}'" if ext == ".tar.zst"
                       else f"tar -xzf '{arquivo}' -C '{PREFIXO_OLLAMA}'")
            _rodar(extrair, "Extração do Ollama")
        except RuntimeError as erro:
            erros.append(str(erro))
            continue
        finally:
            arquivo.unlink(missing_ok=True)
        if ollama_instalado_por_inteiro():
            saida = subprocess.run([str(PREFIXO_OLLAMA / "bin" / "ollama"), "--version"],
                                   capture_output=True, text=True).stdout
            versao = (re.findall(r"\d+\.\d+\.\d+", saida) or ["?"])[-1]
            print(f"✅ Ollama {versao} instalado · llama-server em {localizar_llama_server()}")
            return
        erros.append(f"pacote de {url} veio sem o llama-server")
    raise RuntimeError("Não foi possível instalar o Ollama local:\n- " + "\n- ".join(erros))


def ollama_local_no_ar() -> bool:
    try:
        return httpx.get(EMB_HOST, timeout=2).status_code == 200
    except Exception:
        return False


def iniciar_ollama_local() -> None:
    """Garante o Ollama instalado por inteiro, o servidor local no ar e o nomic-embed-text baixado."""
    env_local = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"}
    env_local.pop("OLLAMA_API_KEY", None)        # o servidor local não usa a chave da nuvem
    binario = str(PREFIXO_OLLAMA / "bin" / "ollama")

    if not ollama_instalado_por_inteiro():       # ausente OU pela metade (binário sem llama-server)
        instalar_ollama()
    if not ollama_local_no_ar():
        subprocess.Popen([binario, "serve"], env=env_local, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(60):
            if ollama_local_no_ar():
                break
            time.sleep(1)
        else:
            raise RuntimeError("O servidor Ollama local não respondeu em 60 s.")
    r = subprocess.run([binario, "pull", MODELO_EMBEDDING], env=env_local, capture_output=True, text=True)
    if r.returncode != 0:
        mensagem = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", r.stderr or r.stdout).strip()   # tira os códigos de cor do terminal
        raise RuntimeError(f"Falha ao baixar o {MODELO_EMBEDDING}:\n{mensagem[-800:]}")


iniciar_ollama_local()
embeddings = OllamaEmbeddings(model=MODELO_EMBEDDING, base_url=EMB_HOST)

# Teste rápido de conexão
vetor_teste = embeddings.embed_query("teste de conexão")
print(f"✅ Embedding OK — {MODELO_EMBEDDING} local, {len(vetor_teste)} dimensões")
print(f"✅ LLM OK — {MODELO_LLM}: {llm.invoke('Responda apenas: OK').content.strip()[:40]}")

In [ ]:
def com_retentativas(funcao, *args, tentativas: int = 5, espera: float = 3.0, **kwargs):
    """Executa uma chamada ao Ollama Cloud com novas tentativas (protege contra limite de taxa e falhas de rede)."""
    for tentativa in range(1, tentativas + 1):
        try:
            return funcao(*args, **kwargs)
        except Exception as erro:
            if tentativa == tentativas:
                raise
            pausa = espera * 2 ** (tentativa - 1)
            print(f"   ⚠️ {type(erro).__name__}: {str(erro)[:80]} — nova tentativa em {pausa:.0f}s")
            time.sleep(pausa)


def slug(texto: str) -> str:
    """Converte um texto em identificador seguro (sem acento, espaço ou símbolo)."""
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto).strip("_").lower()
    return texto or "dominio"

## 3. Base de conhecimento oficial do ENEM (LOAD)

Base **100% oficial (INEP)**. Cada arquivo é descrito no manifesto **`fontes.csv`** (mesmo formato do CKP02, com a
coluna `categoria` trocada por **`materia`**):

| coluna | exemplo | uso |
|---|---|---|
| `arquivo` | `cartilha_redacao_enem_2025.pdf` | nome do arquivo em `docs/` |
| `titulo` | `A Redação no Enem 2025 — Cartilha do Participante` | aparece na citação da resposta |
| `tipo` | `matriz`, `edital`, `cartilha`, `proposta`, `caderno`, `gabarito`, `original_ocr` | filtro `where=` |
| `materia` | `redacao`, `matematica`, `hist_geo`, `biologia`, `fisica`, `geral` ou várias com `\|` (`biologia\|fisica`) | **filtro `where=` de cada agente** |
| `data` | `2025-10-01` | data de publicação |
| `url_fonte` | `https://www.gov.br/inep/...` | link oficial (citado pelo agente) |

Dois caminhos de entrada:
1. **Documentos de referência** (Matriz, Edital, Cartilha da Redação) → PyMuPDF, 1 página = 1 `Document` (código do CKP02).
   > ⚠️ **Matriz de Referência e Cartilha da Redação:** os dois PDFs oficiais do INEP usam fontes sem mapeamento
   > Unicode. Medimos: **~49% do texto da Matriz** sai como símbolos ilegíveis, e a Cartilha tem **~500 linhas
   > ilegíveis em 57 das 78 páginas** (ex.: `$V DXODV VREUH HVFUDYLGmR` no lugar de "As aulas sobre escravidão").
   > Indexar isso inutilizaria a busca. Por isso a base usa o texto desses mesmos PDFs obtido por **OCR** (tesseract,
   > português, 300 dpi — scripts `scripts/gerar_matriz_md.py` e `scripts/gerar_cartilha_md.py`), salvo em `.md`. A
   > Matriz foi separada em um arquivo por matéria. Os marcadores `<!-- pagina: N -->` mantêm a página do PDF
   > original, então a citação aponta para o documento oficial (os PDFs originais ficam em `docs/` como `original_ocr`).
2. **Questões reais** das provas 2024 e 2025 → `dados/banco_questoes_enem.json`. Cada questão vira **1 `Document`**
   com ano, número, página do caderno, matéria e gabarito oficial. Não fatiamos o caderno como texto corrido porque o
   corte cairia no meio de enunciados e alternativas, e porque os cadernos de Ciências da Natureza misturam Biologia,
   Física e Química — a matéria precisa ser marcada **por questão**, não por arquivo.

In [ ]:
# CKP02 usava "categoria"; no CKP03 a coluna vira "materia" — é ela que alimenta o filtro where= de cada agente.
COLUNAS_FONTES = ["arquivo", "titulo", "tipo", "materia", "data", "url_fonte"]
EXTENSOES_ACEITAS = {".pdf", ".txt", ".md"}
MATERIAS_VALIDAS = set(MATERIAS) | {MATERIA_GERAL}
# Arquivos que ficam na base como ORIGEM citável, mas não são indexados como texto corrido:
# - caderno/gabarito: origem do banco de questões (dados/banco_questoes_enem.json), 1 questão = 1 trecho;
# - original_ocr: PDF oficial com fonte sem mapeamento Unicode (Matriz de Referência) — o texto indexado vem da
#   versão OCR em .md, separada por matéria, que mantém a página original.
TIPOS_NAO_INDEXADOS = {"caderno", "gabarito", "original_ocr"}


def preparar_pasta_documentos() -> None:
    """Garante que docs/ e fontes.csv existam: clona o repositório do grupo ou pede upload no Colab."""
    if REPO_GIT and not (PASTA_PROJETO / ".git").exists():
        shutil.rmtree(PASTA_PROJETO, ignore_errors=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_GIT, str(PASTA_PROJETO)], check=True)
    PASTA_DOCS.mkdir(parents=True, exist_ok=True)

    tem_documentos = any(p.suffix.lower() in EXTENSOES_ACEITAS for p in PASTA_DOCS.iterdir())
    if not tem_documentos and EM_COLAB:
        from google.colab import files
        print("📤 Envie os documentos (PDF/TXT/MD) e o fontes.csv — ou um .zip com tudo:")
        for nome, conteudo in files.upload().items():
            if nome.lower().endswith(".zip"):
                destino_zip = PASTA_PROJETO / nome
                destino_zip.write_bytes(conteudo)
                with zipfile.ZipFile(destino_zip) as z:
                    for membro in z.namelist():
                        nome_base = Path(membro).name
                        if not nome_base:
                            continue
                        destino = ARQ_FONTES if nome_base == "fontes.csv" else PASTA_DOCS / nome_base
                        destino.write_bytes(z.read(membro))
            elif nome == "fontes.csv":
                ARQ_FONTES.write_bytes(conteudo)
            else:
                (PASTA_DOCS / nome).write_bytes(conteudo)


def carregar_manifesto() -> pd.DataFrame:
    """Lê o fontes.csv e valida se cada documento existe e tem link de origem."""
    if not ARQ_FONTES.exists():
        # Cria um manifesto inicial com os arquivos encontrados, para o grupo completar
        arquivos = sorted(p.name for p in PASTA_DOCS.iterdir() if p.suffix.lower() in EXTENSOES_ACEITAS)
        pd.DataFrame({"arquivo": arquivos}).reindex(columns=COLUNAS_FONTES).to_csv(ARQ_FONTES, index=False)
        print(f"⚠️ fontes.csv não existia — criei um modelo em {ARQ_FONTES}. Preencha título, tipo, matéria, data e link.")

    manifesto = pd.read_csv(ARQ_FONTES, dtype=str).fillna("")
    faltando = [c for c in COLUNAS_FONTES if c not in manifesto.columns]
    if faltando:
        raise ValueError(f"fontes.csv sem as colunas: {faltando}")
    manifesto = manifesto[COLUNAS_FONTES].apply(lambda col: col.str.strip())

    inexistentes = [a for a in manifesto["arquivo"] if not (PASTA_DOCS / a).exists()]
    if inexistentes:
        raise FileNotFoundError(f"Arquivos listados no fontes.csv mas ausentes em docs/: {inexistentes}")
    fora_do_manifesto = sorted(
        p.name for p in PASTA_DOCS.iterdir()
        if p.suffix.lower() in EXTENSOES_ACEITAS and p.name not in set(manifesto["arquivo"])
    )
    if fora_do_manifesto:
        print(f"⚠️ Arquivos em docs/ fora do fontes.csv (serão ignorados): {fora_do_manifesto}")
    sem_link = manifesto.loc[manifesto["url_fonte"] == "", "arquivo"].tolist()
    if sem_link:
        print(f"⚠️ Documentos sem link de origem (o agente precisa citar a fonte): {sem_link}")
    invalidas = sorted({m for valor in manifesto["materia"] for m in valor.split("|")} - MATERIAS_VALIDAS)
    if invalidas:
        raise ValueError(f"Matéria desconhecida no fontes.csv: {invalidas}. Use uma de {sorted(MATERIAS_VALIDAS)}.")
    return manifesto


preparar_pasta_documentos()
manifesto = carregar_manifesto()

# Tabela de fontes com links (exigência: links de origem no notebook)
linhas = ["| # | Documento | Tipo | Matéria | Data | Fonte |", "|---|---|---|---|---|---|"]
for i, d in enumerate(manifesto.itertuples(), 1):
    link = f"[link]({d.url_fonte})" if d.url_fonte else "—"
    materia = " + ".join(MATERIAS.get(m, {}).get("nome", "todas") for m in d.materia.split("|"))
    linhas.append(f"| {i} | {d.titulo or d.arquivo} | {d.tipo} | {materia} | {d.data} | {link} |")
display(Markdown("\n".join(linhas)))

In [ ]:
def limpar_texto(texto: str) -> str:
    """Normaliza o texto extraído do PDF sem destruir a estrutura de parágrafos (código do CKP02)."""
    texto = texto.replace("\x00", "")
    texto = re.sub(r"(\w)-\n([a-zà-ú])", r"\1\2", texto)   # junta palavras hifenizadas na quebra de linha
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r" ?\n ?", "\n", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


def metadados_base(linha) -> dict:
    """Metadados do manifesto que acompanham cada página e cada chunk (o ChromaDB não aceita None)."""
    return {
        "fonte": linha.arquivo,
        "titulo": linha.titulo or linha.arquivo,
        "tipo": linha.tipo or "nao_informado",
        "materia": linha.materia or MATERIA_GERAL,
        "data": linha.data or "",
        "url_fonte": linha.url_fonte or "",
    }


MARCADOR_PAGINA = re.compile(r"<!--\s*pagina:\s*(\d+)\s*-->")


def paginas_de_texto(texto: str) -> list[tuple[int, str]]:
    """TXT/MD: se houver marcadores <!-- pagina: N -->, cada trecho mantém a página do PDF ORIGINAL (para citar)."""
    partes = MARCADOR_PAGINA.split(texto)
    if len(partes) == 1:
        return [(1, texto)]
    cabecalho = partes[0].strip()
    return [(int(num), f"{cabecalho}\n\n{conteudo}".strip()) for num, conteudo in zip(partes[1::2], partes[2::2])]


def carregar_documento(linha) -> list[Document]:
    """LOAD de um documento de referência: PDF com PyMuPDF (uma página = um Document) ou TXT/MD.
    Um documento pode servir a mais de uma matéria (materia = 'biologia|fisica'): a página é registrada para cada uma."""
    caminho = PASTA_DOCS / linha.arquivo
    meta = metadados_base(linha)
    if caminho.suffix.lower() == ".pdf":
        with fitz.open(caminho) as pdf:
            brutas = [(numero, pagina.get_text()) for numero, pagina in enumerate(pdf, start=1)]
    else:
        brutas = paginas_de_texto(caminho.read_text(encoding="utf-8", errors="ignore"))
    paginas = []
    for numero, bruto in brutas:
        texto = limpar_texto(MARCADOR_PAGINA.sub("", bruto))
        if len(texto) < 30:        # descarta páginas vazias (capa em branco, imagem sem texto)
            continue
        for materia in meta["materia"].split("|"):
            paginas.append(Document(page_content=texto, metadata={**meta, "materia": materia, "pagina": numero}))
    return paginas


def texto_da_questao(q: dict, com_gabarito: bool = True) -> str:
    """Formata uma questão do banco como texto legível (usado no RAG e, sem gabarito, na tool de prática)."""
    alternativas = "\n".join(f"{letra}) {texto}" for letra, texto in q["alternativas"].items())
    texto = f"ENEM {q['ano']} · {q['dia']}º dia · Questão {q['numero']}\n\n{q['enunciado']}\n\n{alternativas}"
    if q.get("tem_imagem"):
        texto += f"\n\n[Esta questão tem figura/gráfico — ver caderno original, p. {q['pagina']}]"
    if com_gabarito:
        texto += f"\n\nGabarito oficial (INEP): {'ANULADA' if q.get('anulada') else q['gabarito']}"
    return texto


def carregar_questoes(manifesto: pd.DataFrame) -> list[dict]:
    """Lê o banco de questões reais e confere se cada questão aponta para um caderno listado no fontes.csv."""
    if not ARQ_QUESTOES.exists():
        print(f"⚠️ {ARQ_QUESTOES.name} não encontrado — o RAG segue só com os documentos de referência.")
        return []
    questoes = json.loads(ARQ_QUESTOES.read_text(encoding="utf-8"))
    cadernos = set(manifesto.loc[manifesto["tipo"] == "caderno", "arquivo"])
    orfas = sorted({q["fonte_arquivo"] for q in questoes} - cadernos)
    if orfas:
        raise ValueError(f"Questões apontam para cadernos fora do fontes.csv: {orfas}")
    invalidas = sorted({q["materia"] for q in questoes} - set(MATERIAS))
    if invalidas:
        raise ValueError(f"Matéria desconhecida no banco de questões: {invalidas}")
    return questoes


def questoes_para_documentos(questoes: list[dict], manifesto: pd.DataFrame) -> list[Document]:
    """Cada questão vira 1 Document, com os metadados do caderno de origem + número, ano e página."""
    por_arquivo = {linha.arquivo: linha for linha in manifesto.itertuples()}
    documentos = []
    for q in questoes:
        caderno = por_arquivo[q["fonte_arquivo"]]
        meta = {**metadados_base(caderno),
                "titulo": f"{caderno.titulo} — Questão {q['numero']}",
                "tipo": "questao", "materia": q["materia"], "pagina": int(q["pagina"]),
                "ano": int(q["ano"]), "numero": int(q["numero"]), "questao_id": q["id"]}
        documentos.append(Document(page_content=texto_da_questao(q), metadata=meta))
    return documentos


def carregar_base(manifesto: pd.DataFrame) -> tuple[list[Document], list[dict]]:
    """LOAD da base inteira: documentos de referência (por página) + questões reais (por questão)."""
    todas, resumo = [], []
    for linha in manifesto.itertuples():
        if linha.tipo in TIPOS_NAO_INDEXADOS:
            continue                       # cadernos/gabaritos entram via banco de questões; PDF original via .md
        paginas = carregar_documento(linha)
        todas.extend(paginas)
        resumo.append({"documento": linha.titulo, "matéria": linha.materia, "páginas com texto": len(paginas),
                       "caracteres": sum(len(p.page_content) for p in paginas)})
        if not paginas:
            print(f"⚠️ {linha.arquivo}: nenhum texto extraído (PDF escaneado?). Troque por uma versão com texto.")

    questoes = carregar_questoes(manifesto)
    docs_questoes = questoes_para_documentos(questoes, manifesto)
    todas.extend(docs_questoes)
    for materia, info in MATERIAS.items():
        qtd = sum(1 for q in questoes if q["materia"] == materia)
        resumo.append({"documento": f"Questões reais ENEM 2024–2025 · {info['nome']}", "matéria": materia,
                       "páginas com texto": f"{qtd} questões",
                       "caracteres": sum(len(d.page_content) for d in docs_questoes if d.metadata["materia"] == materia)})
    display(pd.DataFrame(resumo))
    return todas, questoes


paginas, BANCO_QUESTOES = carregar_base(manifesto)
print(f"\nTotal: {len(paginas)} documentos (páginas + questões) · {sum(len(p.page_content) for p in paginas):,} caracteres")

## 4. Chunking (SPLIT)

Usamos direto a **configuração vencedora do CKP02** — `RecursiveCharacterTextSplitter` com `chunk_size=1024` e
`chunk_overlap=128` (12,5%). No CKP02 ela venceu o RAGAS (faithfulness 0,771 contra 0,698 do 512) porque mantém regra
e exceção no mesmo trecho — o mesmo padrão da Cartilha da Redação ("o que zera", "o que tira pontos em cada competência").

- **Documentos de referência** → fatiados em chunks de 1024.
- **Questões** → **não são fatiadas**: 1 questão = 1 chunk. Um enunciado cortado ao meio, separado das alternativas e
  do gabarito, não serve nem para o RAG nem para a prática.

In [ ]:
def gerar_chunks(documentos: list[Document]) -> list[Document]:
    """SPLIT: fatia os documentos de referência e mantém cada questão inteira. Cria um id estável para cada chunk."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=SEPARADORES)
    referencia = [d for d in documentos if d.metadata["tipo"] != "questao"]
    questoes   = [d for d in documentos if d.metadata["tipo"] == "questao"]

    chunks = [c for c in splitter.split_documents(referencia) if len(c.page_content.strip()) >= TAMANHO_MINIMO_CHUNK]
    for i, chunk in enumerate(chunks):
        assinatura = f"{chunk.metadata['fonte']}|{chunk.metadata['materia']}|{chunk.metadata['pagina']}|{i}|{chunk.page_content}"
        chunk.metadata["chunk_id"] = f"cs{CHUNK_SIZE}-{hashlib.md5(assinatura.encode()).hexdigest()[:12]}"
    for q in questoes:
        q.metadata["chunk_id"] = f"q-{q.metadata['questao_id']}"
    for c in chunks + questoes:
        c.metadata["chunk_size"] = CHUNK_SIZE
    return chunks + questoes


chunks = gerar_chunks(paginas)

estatisticas = (pd.DataFrame([{"matéria": c.metadata["materia"], "tipo": c.metadata["tipo"], "tamanho": len(c.page_content)}
                              for c in chunks])
                .groupby(["matéria", "tipo"])["tamanho"].agg(chunks="count", tamanho_medio="mean").round(0).reset_index())
display(estatisticas)
print(f"Total: {len(chunks)} chunks")

## 5. EMBED + STORE — uma coleção ChromaDB, filtrada por matéria

Uma **única coleção** (`tutor_enem_cs1024`, distância cosseno). Cada agente enxerga só a sua fatia via
`where={"materia": {"$in": [<matéria>, "geral"]}}` — o filtro é aplicado **antes** do cálculo de similaridade, então o
tutor de Física nunca recebe trecho de Redação. Uma coleção só evita duplicar os documentos "geral" (Matriz, Edital) em
5 lugares e deixa a 6ª matéria a um `fontes.csv` de distância.

Os vetores são gerados no nosso código (`embed_documents`) e o ChromaDB só armazena e busca (Aula 05). Se a base não
mudou desde a última execução, a coleção é reaproveitada (assinatura MD5 do manifesto + banco de questões + parâmetros).

In [ ]:
cliente_chroma = chromadb.PersistentClient(path=str(PASTA_CHROMA))
NOME_COLECAO = f"{slug(DOMINIO)}_cs{CHUNK_SIZE}"        # tutor_enem_cs1024


def adicionar_chunks(colecao, chunks: list[Document], lote: int = 64) -> None:
    """EMBED + STORE: gera os embeddings em lotes e grava texto, vetor, id e metadados no ChromaDB (código do CKP02)."""
    for inicio in range(0, len(chunks), lote):
        if inicio and inicio % (lote * 25) == 0:
            print(f"   … {inicio}/{len(chunks)} chunks", flush=True)
        bloco = chunks[inicio:inicio + lote]
        vetores = com_retentativas(embeddings.embed_documents, [c.page_content for c in bloco])
        colecao.add(
            ids=[c.metadata["chunk_id"] for c in bloco],
            documents=[c.page_content for c in bloco],
            embeddings=vetores,
            metadatas=[c.metadata for c in bloco],
        )


def assinatura_base(chunks: list[Document]) -> str:
    """Impressão digital da base + parâmetros: se nada mudou, a coleção já indexada é reaproveitada."""
    banco = ARQ_QUESTOES.read_text(encoding="utf-8") if ARQ_QUESTOES.exists() else ""
    conteudo = "|".join([manifesto.to_csv(index=False), hashlib.md5(banco.encode()).hexdigest(), str(CHUNK_SIZE),
                         str(CHUNK_OVERLAP), str(SEPARADORES), MODELO_EMBEDDING, str(len(chunks))])
    return hashlib.md5(conteudo.encode()).hexdigest()


def indexar(chunks: list[Document], forcar: bool = False):
    """Cria a coleção e indexa os chunks (reaproveita se a base não mudou)."""
    assinatura = assinatura_base(chunks)
    if NOME_COLECAO in [c.name for c in cliente_chroma.list_collections()]:
        existente = cliente_chroma.get_collection(NOME_COLECAO)
        if not forcar and (existente.metadata or {}).get("assinatura") == assinatura and existente.count() == len(chunks):
            print(f"♻️ {NOME_COLECAO}: base não mudou, reaproveitando {existente.count()} chunks já indexados")
            return existente
        cliente_chroma.delete_collection(NOME_COLECAO)
    colecao = cliente_chroma.create_collection(
        name=NOME_COLECAO,
        metadata={"hnsw:space": "cosine", "dominio": DOMINIO, "chunk_size": CHUNK_SIZE, "assinatura": assinatura},
    )
    adicionar_chunks(colecao, chunks)
    return colecao


iniciar_ollama_local()          # garante o servidor de embeddings no ar antes de indexar
inicio = time.time()
COLECAO = indexar(chunks)
print(f"✅ {NOME_COLECAO}: {COLECAO.count()} chunks prontos em {time.time() - inicio:.0f}s")

# Conferência: quantos chunks cada agente enxerga (os da matéria + os "geral")
display(pd.DataFrame([{"agente": f"{info['emoji']} {info['nome']}",
                       "chunks visíveis": len(COLECAO.get(where={"materia": {"$in": [m, MATERIA_GERAL]}}, include=[])["ids"])}
                      for m, info in MATERIAS.items()]))

## 6. RAG como tool — `create_retriever_tool()` (Aula 11, Passo 1)

O pipeline do CKP02 vira uma **tool nativa do agente**:

1. `Chroma(client=..., collection_name=...)` abre a **mesma coleção** indexada na Seção 5 como vector store do LangChain.
2. `as_retriever(filter=...)` aplica o **filtro por matéria** (metadata filtering do CKP02).
3. `ContextualCompressionRetriever` + `CrossEncoderReranker` reaproveitam o **reranking** do CKP02
   (`ms-marco-MiniLM-L-6-v2`): busca 12 candidatos e entrega os 4 melhores.
4. `create_retriever_tool()` encapsula tudo. O `document_prompt` põe **título, página e link** em cima de cada trecho —
   é isso que permite ao agente **citar a fonte** (critério de 2,5 pontos).

Uma fábrica, `criar_tool_rag(materia)`, gera a tool de cada agente: **mesmo nome** (`buscar_material_enem`), **mesmo
código**, mas **description e filtro próprios** da matéria.

> 📝 **Sobre o wording das descriptions** (diferencial da Aula 11): a description é o roteador do ReAct — o LLM só
> enxerga ela para decidir. Por isso cada uma tem 4 partes fixas, comentadas no código: **(1) o que tem na base**,
> **(2) quando usar**, com exemplos reais de pergunta da matéria, **(3) o que devolve** (e a ordem de citar) e
> **(4) quando NÃO usar** — apontando para onde a pergunta deve ir (conhecimento próprio, outra tool).

In [ ]:
from langchain_chroma import Chroma
# Aula 11: `from langchain.tools.retriever import create_retriever_tool` → no LangChain 1.x está no langchain_classic
from langchain_classic.tools.retriever import create_retriever_tool
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# A mesma coleção da Seção 5, agora como vector store do LangChain (os embeddings da pergunta usam o mesmo nomic-embed-text)
vectorstore = Chroma(client=cliente_chroma, collection_name=NOME_COLECAO, embedding_function=embeddings)


def filtro_materia(materia: str) -> dict:
    """Metadata filtering: cada agente vê só os documentos da sua matéria + os documentos gerais do ENEM."""
    return {"materia": {"$in": [materia, MATERIA_GERAL]}}


_cross_encoder = None


def obter_cross_encoder():
    """Carrega o cross-encoder do CKP02 só uma vez (é compartilhado pelos 5 agentes)."""
    global _cross_encoder
    if _cross_encoder is None:
        _cross_encoder = HuggingFaceCrossEncoder(model_name=MODELO_RERANKER)
    return _cross_encoder


def criar_retriever(materia: str, rerank: bool = True):
    """Retriever da matéria: busca vetorial filtrada + (opcional) reranking por cross-encoder."""
    if not rerank:
        return vectorstore.as_retriever(search_kwargs={"k": K_FINAL, "filter": filtro_materia(materia)})
    try:
        base = vectorstore.as_retriever(search_kwargs={"k": K_CANDIDATOS, "filter": filtro_materia(materia)})
        compressor = CrossEncoderReranker(model=obter_cross_encoder(), top_n=K_FINAL)
        return ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base)
    except Exception as erro:   # sem o reranker o RAG continua funcionando (só a busca vetorial)
        print(f"⚠️ Reranking indisponível ({type(erro).__name__}); {materia} segue só com a busca vetorial.")
        return vectorstore.as_retriever(search_kwargs={"k": K_FINAL, "filter": filtro_materia(materia)})


# Cabeçalho de cada trecho na Observation → é daqui que o agente tira a citação "(Fonte: ..., p. ...)"
DOCUMENT_PROMPT = PromptTemplate.from_template("[Fonte: {titulo}, p. {pagina} — {url_fonte}]\n{page_content}")

# ===================== DESCRIPTIONS (o roteador do ReAct) =====================
# Exemplos REAIS de pergunta por matéria. Wording: perguntas sobre "o que o ENEM cobra / como avalia / o que já caiu",
# que é exatamente o que só a base oficial responde. Sem exemplos, o gemma tende a usar o RAG para TUDO
# (inclusive "explique o que é mitose"), gastando uma chamada e trazendo questão em vez de explicação.
EXEMPLOS_RAG = {
    "redacao":    ["o que zera a redação no ENEM?",
                   "o que a Cartilha diz sobre a Competência 5 e a proposta de intervenção?",
                   "quais foram os temas da redação em 2024 e 2025?"],
    "matematica": ["que questões de porcentagem ou estatística caíram no ENEM 2024?",
                   "quais habilidades a Matriz de Referência cobra em Matemática?"],
    "hist_geo":   ["que questões sobre Brasil Colônia ou Era Vargas caíram no ENEM?",
                   "quais competências de Ciências Humanas a Matriz de Referência cobra?"],
    "biologia":   ["que questões de genética ou ecologia caíram no ENEM 2025?",
                   "o que a Matriz de Referência cobra sobre saúde e meio ambiente?"],
    "fisica":     ["que questões de eletricidade ou energia caíram no ENEM?",
                   "quais habilidades de Física aparecem na Matriz de Referência?"],
}

CONTEUDO_BASE = {
    # (1) O QUE TEM NA BASE — dito explicitamente para o LLM não esperar livro didático ali dentro
    "redacao": "a Cartilha do Participante 'A Redação no Enem 2025' (5 competências, o que zera a nota, redações nota "
               "alta comentadas), a Matriz de Referência, o Edital do ENEM 2026 e as propostas de redação de 2024 e 2025",
}


def descricao_rag(materia: str) -> str:
    info = MATERIAS[materia]
    base = CONTEUDO_BASE.get(materia, f"a Matriz de Referência, o Edital do ENEM 2026 e as questões REAIS de "
                                      f"{info['nome']} das provas de 2024 e 2025, com gabarito oficial")
    exemplos = "; ".join(f"'{e}'" for e in EXEMPLOS_RAG[materia])
    return (
        # (1) o que tem na base — nomeia o domínio (troubleshooting da Aula 11: description genérica = tool errada)
        f"Busca nos documentos OFICIAIS do ENEM (INEP) de {info['nome']}: {base}. "
        # (2) quando usar — verbos do aluno + exemplos reais
        f"Use quando o aluno perguntar o que o ENEM cobra, como o ENEM avalia, quais as regras oficiais da prova "
        f"ou que questões já caíram sobre um tema de {info['nome']}. Exemplos: {exemplos}. "
        # (3) o que devolve — e a obrigação de citar
        f"Retorna trechos com [Fonte: título, página e link]: cite a fonte na resposta final. "
        # (4) quando NÃO usar — manda a pergunta para o lugar certo
        f"NÃO use para explicar um conceito de {info['nome']} (responda com seu próprio conhecimento), "
        f"para cálculos, para sortear questão de treino nem para notícias atuais."
    )


def criar_tool_rag(materia: str, rerank: bool = True):
    """Fábrica da tool RAG: mesmo nome e mesmo código para todos os agentes; description e filtro da matéria."""
    return create_retriever_tool(
        criar_retriever(materia, rerank=rerank),
        name="buscar_material_enem",
        description=descricao_rag(materia),
        document_prompt=DOCUMENT_PROMPT,
        document_separator="\n\n---\n\n",
    )


TOOLS_RAG = {materia: criar_tool_rag(materia) for materia in MATERIAS}
print(f"✅ {len(TOOLS_RAG)} tools RAG criadas: {', '.join(f'{m} → {t.name}' for m, t in TOOLS_RAG.items())}")
print(f"\nExemplo de description (Física):\n{TOOLS_RAG['fisica'].description}")

### 6.1 Teste isolado da tool RAG (antes de montar o agente)

Regra da Aula 11: **toda tool é testada sozinha com `invoke()`** antes de entrar no agente. Se o agente errar depois,
sabemos que o problema é de decisão (description/prompt), não da tool. O teste também confere o **isolamento por
matéria**: nenhum trecho de outra matéria pode aparecer.

In [ ]:
PERGUNTAS_TESTE_RAG = {materia: exemplos[0] for materia, exemplos in EXEMPLOS_RAG.items()}

resultados = []
for materia, pergunta in PERGUNTAS_TESTE_RAG.items():
    inicio = time.time()
    observation = TOOLS_RAG[materia].invoke(pergunta)                    # o que o agente vai receber
    trechos = criar_retriever(materia, rerank=False).invoke(pergunta)    # conferência do filtro
    vazou = [t.metadata["materia"] for t in trechos if t.metadata["materia"] not in (materia, MATERIA_GERAL)]
    assert not vazou, f"Filtro falhou em {materia}: vieram trechos de {vazou}"
    fontes = re.findall(r"\[Fonte: (.*?) —", observation)
    resultados.append({"agente": MATERIAS[materia]["nome"], "pergunta": pergunta, "chars": len(observation),
                       "fontes citáveis": len(fontes), "1ª fonte": fontes[0] if fontes else "—",
                       "tempo (s)": round(time.time() - inicio, 1)})

display(pd.DataFrame(resultados))
print("✅ Filtro por matéria OK: nenhum trecho de outra matéria apareceu.\n")
print("Observation completa que o agente de Redação recebe:\n")
print(TOOLS_RAG["redacao"].invoke(PERGUNTAS_TESTE_RAG["redacao"])[:1500])